# ملاحظات و یادداشت ها

## تنوع در داده های آزمایش؟!
- train

  θ∈[0,π]

  noise -->  p∈[0.05,0.15]

- test

  θ∈[0,π/2]

  noise -->  p∈[0.1,0.25]


اگر مدل سازی یک سخت افزار کوانتومی مشخص باشد، شاید لازم باشد که توزیع داده های آموزشی و ازمایشی یکسان باشد. اما 

در مدارهای کوانتومی رایج است که برای سخت افزار مقداری عدم ثبات در نظر بگیرند، چون 

- نویز ثابت نیست

- کالیبراسیون متفاوت است

- خوانش روز به روز عوض می شود

- دما و الکترونیک یکسان نیست

  

## مدل فیزیکی و تفاوت آن با مدل معمولی شبکه عصبی

### مقایسه مدل‌های `model_mlp` و `model_phys` در کاهش خطای خوانش کوانتومی


 هدف کاهش خطای خوانش کاهش خطا در یک سیستم کوانتومی چهارکیوبیتی شبیه‌سازی‌شده با کیسکیت است. برای این منظور، دو رویکرد متفاوت مبتنی بر یادگیری ماشین بررسی شده‌اند:

1. مدل داده‌محور (`model_mlp`)
2. مدل فیزیک‌محور (`model_phys`)

اگرچه هر دو مدل در نهایت تلاش می‌کنند توزیع احتمالات ایده‌آل را از داده‌های نویزی بازسازی کنند، اما فلسفه عملکرد، ساختار و نحوه یادگیری آن‌ها کاملاً متفاوت است.

---

#### مدل اول: `model_mlp`

### ایده اصلی

در این مدل، شبکه عصبی مستقیماً رابطه بین توزیع نویزی و توزیع ایده‌آل را یاد می‌گیرد:

\[
p_{noisy} \rightarrow p_{ideal}
\]

یا به صورت تابع:

\[
f_{NN}(p_{noisy}) \approx p_{ideal}
\]

در این روش، شبکه عصبی بدون درنظر گرفتن ساختار فیزیکی نویز، صرفاً یک نگاشت میان ورودی و خروجی یاد می‌گیرد.

---

#### طرحواره عملکرد مدل

```text
Qiskit Noise Model
        ↓
  Noisy Distribution
        ↓
      MLP
        ↓
Estimated Ideal Distribution 
```

ویژگی های مدل:

- کاملاً داده‌محور

- یادگیری مستقیم نگاشت

- ساختار نویز در شبکه پنهان می‌ماند

این مدل شبیه شبکه ای هست که تصویر نویزی تحویل می گیرد و تصویر تمیز تحویل می دهد بدون این که ساختار نویز را بفهد. 

---

### مدل دوم: `model_phys`

#### ایده اصلی

ابتدا خودِ نویز مدل‌سازی می‌شود:

M_learned ≈ M_true

ماتریس نویز تک کیوبیت:

M =
[[1-p01, p01],
 [p10, 1-p10]]

و برای ۴ کیوبیت:

M_global = M1 ⊗ M2 ⊗ M3 ⊗ M4

---

#### مرحله دوم: mitigation

p_noisy = M p_ideal  
p_ideal ≈ M^{-1} p_noisy

---

#### طرحواره عملکرد

```
Qiskit Noise Model
        ↓
  Noisy Distribution
        ↓
 Neural Network
        ↓
 Learned Noise Matrix (M)
        ↓
 Physical Inversion (M⁻¹)
        ↓
Estimated Ideal Distribution

```
---

#### ویژگی‌ها
- Physics-informed
- قابل تفسیر
- یادگیری نویز به جای جواب مستقیم

این مدل شبیه حالتی است که برای پلایش تصویر نویزی، مکانیزم نویز را می فهمد  سپس با استفاده از مدل فیزیکی نویز، تصویر اصلاح گرد.

---

### تفاوت اصلی

model_mlp:
- مستقیم noisy → ideal

model_phys:
- noisy → M → inversion → ideal

---

### شاخص‌های ارزیابی

Fidelity:
F(p,q) = (Σ sqrt(p_i q_i))^2

L1:
Σ |p_i - q_i|

KL:
Σ P(i) log(P(i)/Q(i))

---

### جمع‌بندی

MLP: سریع و مستقیم است اما فیزیک را نمی‌فهمد.  
model_phys: ساختار نویز را یاد می‌گیرد و mitigation فیزیکی انجام می‌دهد.


کیسکین معمولا نویزها را به دو دسته تقسیم می‌کند:

 Noise channels (کانال‌های نویز روی کیوبیت‌ها یا گیت‌ها)
1. Depolarizing channel: تصادفی باعث flip شدن کیوبیت‌ها می‌شود.

Amplitude damping: مدل فروپاشی انرژی (T1 decay)

Phase damping / dephasing: مدل کاهش coherence (T2 decay)

Pauli errors: X, Y, Z probabilistic flips

4. Measurement noise (Readout error)